In [1]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER
import cftime
import datetime
from datetime import date
from matplotlib import pyplot
import numpy
import pandas
import xarray as xr

In [2]:
Diri = '../ExtraTrack_Data/RCP_Files_V7/'
Output_Diri = '../ExtraTrack_Data/Output_Files_V7/'

In [3]:
# Open File
def Create_DF(File):
    Data = open(File, 'r')
    Rows = []
    Storm_Code, Storm_List_Orig = [], []
#
# Get Storm Codes
    for Line in Data:
        Line = Line.strip()
        if Line.startswith("start"):
            Code = Line.split()[-1]
            Storm_List_Orig.append(Code)
        else:
            Storm_Code.append(Code)
            Values = Line.split()
            Rows.append(Values)
#
# Create Dataframe
    Cols = ["Lon", "Lat", "SLP(hPa)", "Winds(m/s)", "Dist(m)", "Angle", "B", "VLT", "VUT", "Year", "Month", "Day", "Hour"]
    DF = pandas.DataFrame(Rows, columns=Cols)
    DF = DF.apply(pandas.to_numeric)
    DF["Time(Z)"] = pandas.to_datetime(DF[["Year", "Month", "Day", "Hour"]])
    DF = DF.drop(columns=["Year", "Month", "Day", "Hour"])
    DF.insert(0, "Code", Storm_Code)
#
# Remove Data Points Missing Phase Space Parameter Values (-999.0) and SLP > 1013hPa
    Mask = (DF != -999.00).all(axis=1)
    DF = DF[Mask].reset_index(drop=True)
    DF = DF[DF["SLP(hPa)"] <= 1013].reset_index(drop=True)
    return (DF, Storm_List_Orig)

In [4]:
# Combine The Three Datasets Into One DataFrame of 90 or 93 Years For Each Scenario
def Combine_DF(DF_A, DF_B, DF_C, Orig_List_A, Orig_List_B, Orig_List_C, Model):
# Change Date For Simplicity of Analysis
# Control Assigned to Years in the 20th Century
# Each Year Within the Century are Independent
    if Model == "Control":
        Year_Start = 1900
# RCP4.5 Assigned to Years in the 21st Century
    elif Model == "RCP45":
        Year_Start = 2000
# RCP8.5 Assigned to Years in the 22nd Century
    elif Model == "RCP85":
        Year_Start = 2100
#
# Create Empty Lists
    Orig_Combine, New_Combine, Ensemble = [], [], []
    DF_Combine = DF_A.iloc[0:0].copy()
#
# Dataset A
    Year = Year_Start
    Orig_Year = DF_A["Time(Z)"][0].year
    Year_Diff_A = Year - Orig_Year
    Count = 0
    for k in range(len(Orig_List_A)):
        Orig_Code = Orig_List_A[k]
        DF_Storm = DF_A[DF_A["Code"] == Orig_Code].reset_index(drop=True)
        if len(DF_Storm) > 0:
            New_Code, Year, Count = Update_Code(DF_Storm, Year_Diff_A, Year, Count)
            Orig_Combine.append(Orig_Code)
            New_Combine.append(New_Code)
            Ensemble.append("A")
            DF_Combine = Combine_Lists(DF_Combine, DF_Storm, New_Code, Year_Diff_A)
#
# Dataset B
    Year += 1
    Orig_Year = DF_B["Time(Z)"][0].year
    Year_Diff_B = Year - Orig_Year
    Count = 0
    for k in range(len(Orig_List_B)):
        Orig_Code = Orig_List_B[k]
        DF_Storm = DF_B[DF_B["Code"] == Orig_Code].reset_index(drop=True)
        if len(DF_Storm) > 0:
            New_Code, Year, Count = Update_Code(DF_Storm, Year_Diff_B, Year, Count)
            Orig_Combine.append(Orig_Code)
            New_Combine.append(New_Code)
            Ensemble.append("B")
            DF_Combine = Combine_Lists(DF_Combine, DF_Storm, New_Code, Year_Diff_B)
#
# Dataset C
    Year += 1
    Orig_Year = DF_C["Time(Z)"][0].year
    Year_Diff_C = Year - Orig_Year
    Count = 0
    for k in range(len(Orig_List_C)):
        Orig_Code = Orig_List_C[k]
        DF_Storm = DF_C[DF_C["Code"] == Orig_Code].reset_index(drop=True)
        if len(DF_Storm) > 0:
            New_Code, Year, Count = Update_Code(DF_Storm, Year_Diff_C, Year, Count)
            Orig_Combine.append(Orig_Code)
            New_Combine.append(New_Code)
            Ensemble.append("C")
            DF_Combine = Combine_Lists(DF_Combine, DF_Storm, New_Code, Year_Diff_C)

# Create DataFrame to Store Data
    Codes_DF = pandas.DataFrame({"Ensemble": Ensemble, "Orig Code": Orig_Combine, "New Code": New_Combine})
    return (DF_Combine, Codes_DF)

In [5]:
# Update Storm Code to Contain Year and Storm Number
def Update_Code(DF_Storm, Year_Diff, Year, Count):
    Orig_Year = DF_Storm["Time(Z)"][0].year
    Time_Year = Orig_Year + Year_Diff
    if Time_Year == Year:
        Count += 1
    else:
        Year += 1
        Count = 1
    Code = Code = f"TC{Year}{Count:02d}"
    return (Code, Year, Count)

In [6]:
# Combine Lists
def Combine_Lists(DF_Combine, DF_Storm, New_Code, Year_Diff):
    New_Time, New_Codes = [], []
    for i in range(len(DF_Storm)):
        New_Time.append(Update_Year(DF_Storm["Time(Z)"][i], Year_Diff))
        New_Codes.append(New_Code)
    DF_Storm["Code"] = New_Codes
    DF_Storm["Time(Z)"] = New_Time
    DF_Combine = pandas.concat([DF_Combine, DF_Storm], ignore_index=True)
    return (DF_Combine)

In [7]:
# Change Year of Data
def Update_Year(Orig_Time, Year_Diff):
    Year_Update = Orig_Time.year + Year_Diff
# Febuary 29 Issues
    if Orig_Time.month == 2 and Orig_Time.day == 29:
        New_Time = Orig_Time.replace(year=Year_Update, day=28)
    else:
        New_Time = Orig_Time.replace(year=Year_Update)
    return (New_Time)

In [8]:
# Open Name List File
def Open_Name_List(File):
    Name_File = pandas.read_csv(File)
    Headings = list(Name_File.columns)
    Name_List = []
    for i in range(len(Headings)):
        for j in range(len(Name_File)):
            Name_List.append(Name_File[Headings[i]][j])
    return (Name_List)

In [9]:
# Assign Names to Each Cyclone Code
def Assign_Name(Name_List, Data_DF, ET_DF, Codes_DF):
    Storm_Names = []
    for i in range(len(Codes_DF)):
        Name = Name_List[int(i%len(Name_List))]
        Storm_Names.append(Name)
    Codes_DF["Name"] = Storm_Names
    ET_DF.insert(1, "Name", Storm_Names)
    DF_Names = []
    for k in range(len(Data_DF)):
        Code = Data_DF["Code"][k]
        Name = list(Codes_DF[Codes_DF["New Code"] == Code]["Name"])[0]
        DF_Names.append(Name)
    Data_DF.insert(1, "Name", DF_Names)
    return (Data_DF, ET_DF, Codes_DF)

In [10]:
# Function to Find Distance Between Two Points
def Find_Distance(y1, y2, x1, x2):
    Start_Lat = y1 * numpy.pi / 180
    End_Lat = y2 * numpy.pi / 180
    Start_Lon = x1 * numpy.pi / 180
    End_Lon = x2 * numpy.pi / 180
    Lat_Diff = End_Lat - Start_Lat
    Lon_Diff = End_Lon - Start_Lon
    Earth_Rad = 6378
    Distance = 2 * Earth_Rad * numpy.sqrt((numpy.sin(Lat_Diff/2))**2 + \
    numpy.cos(Start_Lat) * numpy.cos(End_Lat) * (numpy.sin(Lon_Diff/2))**2)
    return (Distance)

In [11]:
# Find a Specific Storm Within the DataFrame
def Find_Storm(DF, Code):
    DF_Storm = DF[DF["Code"] == Code].reset_index(drop=True)
    return (DF_Storm)

In [12]:
# Open Name List
Name_List = Open_Name_List('Storm_Name_List.csv')

In [13]:
# Apply Functions to Open Datasets
Control_A_Data_V0, Control_A_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.control_avg')
Control_B_Data_V0, Control_B_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.control.002_avg')
Control_C_Data_V0, Control_C_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.control.003_avg')

In [14]:
# Process Control DataFrames
Control_Data_V1, Control_Codes_V1 = \
Combine_DF(Control_A_Data_V0, Control_B_Data_V0, Control_C_Data_V0, \
Control_A_Storm_List_V0, Control_B_Storm_List_V0, Control_C_Storm_List_V0, "Control")

In [15]:
# Apply Functions to Open Datasets
RCP45_A_Data_V0, RCP45_A_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.rcp45_avg')
RCP45_B_Data_V0, RCP45_B_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.rcp45.002_avg')
RCP45_C_Data_V0, RCP45_C_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.rcp45.003_avg')

In [16]:
# Process RCP4.5 DataFrames
RCP45_Data_V1, RCP45_Codes_V1 = \
Combine_DF(RCP45_A_Data_V0, RCP45_B_Data_V0, RCP45_C_Data_V0, \
RCP45_A_Storm_List_V0, RCP45_B_Storm_List_V0, RCP45_C_Storm_List_V0, "RCP45")

In [17]:
# Apply Functions to Open Datasets
RCP85_A_Data_V0, RCP85_A_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.rcp85_avg')
RCP85_B_Data_V0, RCP85_B_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.rcp85.003_avg')
RCP85_C_Data_V0, RCP85_C_Storm_List_V0 = Create_DF(Diri + 'traj_et_dtime900.rcp85.004_avg')

In [18]:
# Process RCP8.5 DataFrames
RCP85_Data_V1, RCP85_Codes_V1 = \
Combine_DF(RCP85_A_Data_V0, RCP85_B_Data_V0, RCP85_C_Data_V0, \
RCP85_A_Storm_List_V0, RCP85_B_Storm_List_V0, RCP85_C_Storm_List_V0, "RCP85")

In [19]:
# Create DF With Extratropical Transition Information of Each Storm
def Storm_Info_DF(DF, Codes):
    Ensemble = []
    Trans_Types, Trop_Begin_Times, ET_Begin_Times, ET_Compl_Times, Track_Discons = [], [], [], [], []
    for i in range(len(Codes)):
        DF_Storm_Orig = DF[DF["Code"] == Codes["New Code"][i]].reset_index(drop=True)
        DF_Storm, Track_Discons = DF_Fixing(DF_Storm_Orig, Track_Discons)
        Trans_Type, Trop_Begin_t, ET_Begin_t, ET_Compl_t = ET_Function(DF_Storm)
        if Trop_Begin_t >= 0:
            Trop_Begin_Time = DF_Storm["Time(Z)"][Trop_Begin_t]
        if ET_Begin_t > 0:
            ET_Begin_Time = DF_Storm["Time(Z)"][ET_Begin_t]
        else:
            ET_Begin_Time = numpy.nan
        if ET_Compl_t > 0:
            ET_Compl_Time = DF_Storm["Time(Z)"][ET_Compl_t]
        else:
            ET_Compl_Time = numpy.nan
        Trop_Begin_Times.append(Trop_Begin_Time)
        ET_Begin_Times.append(ET_Begin_Time)
        ET_Compl_Times.append(ET_Compl_Time)
        Trans_Types.append(Trans_Type)
        Ensemble.append(Codes["Ensemble"][i])
        if i == 0:
            DF_Fixed = DF_Storm.copy()
        else:
            DF_Fixed = pandas.concat([DF_Fixed, DF_Storm])
    DF_Fixed = DF_Fixed.reset_index(drop=True)
#
# Create Storms DF
    Storms_DF_Orig = pandas.DataFrame({"Code": Codes["New Code"], "Orig Code": Codes["Orig Code"], "Ensemble": Ensemble, \
    "Trans Type": Trans_Types, "Trop Begin Time": Trop_Begin_Times, "ET Begin Time": ET_Begin_Times, "ET Complete Time": ET_Compl_Times})
    return (DF_Fixed, Storms_DF_Orig)

In [20]:
# Function For Check For Time Discontinuity
def Time_Discontinuity(DF_Storm):
    Time_Continuous = 0
    for k in range(len(DF_Storm)):
        if k > 0 and Time_Continuous == 0:
#            print (k)
# If Over 12 Hours of Missing Data
            if DF_Storm["Time(Z)"][k] - DF_Storm["Time(Z)"][k-1] >= datetime.timedelta(hours=12):
# Divide Array Into Before The Time Discontinuity and After It
                Before = DF_Storm["Time(Z)"][k-1]
                After = DF_Storm["Time(Z)"][k]
                DF_Before = DF_Storm[DF_Storm["Time(Z)"] <= Before].reset_index(drop=True)
                DF_After = DF_Storm[DF_Storm["Time(Z)"] >= After].reset_index(drop=True)
# Choose the Part with the Lowest Min SLP to Keep
                if numpy.min(DF_After["SLP(hPa)"]) <= numpy.min(DF_Before["SLP(hPa)"]) and \
                len(DF_After) > len(DF_Before):
                    DF_Storm_Updated = DF_After
                else:
                    DF_Storm_Updated = DF_Before
                Time_Continuous = 1
# Repeat Until No More Time Discontinuity
    if Time_Continuous == 0:
        DF_Storm_Updated = DF_Storm.copy()
    return (DF_Storm_Updated)

In [21]:
# Function For Check For Track Discontinuity
def Track_Discontinuity(DF_Storm):
    Track_Continuous = 0
    Track_Discon = []
    for k in range(len(DF_Storm)):
        if k > 1 and Track_Continuous == 0:
# Calculate Distance Between Each 6 Hour Interval
            Distance_1 = Find_Distance(DF_Storm["Lat"][k], DF_Storm["Lat"][k-1], \
            DF_Storm["Lon"][k], DF_Storm["Lon"][k-1])
            Distance_2 = Find_Distance(DF_Storm["Lat"][k-1], DF_Storm["Lat"][k-2], \
            DF_Storm["Lon"][k-1], DF_Storm["Lon"][k-2])
# If Distance >= 800km and Distance Over 5 Times Further Than Previous Time Interval
            if Distance_1 >= 800 and Distance_1 >= Distance_2 * 5:
                Track_Discon = [DF_Storm["Code"][0], DF_Storm["Time(Z)"][k]]
# Divide Array Into Before The Track Discontinuity and After It
                Before = DF_Storm["Time(Z)"][k-1]
                After = DF_Storm["Time(Z)"][k]
                DF_Before = DF_Storm[DF_Storm["Time(Z)"] <= Before].reset_index(drop=True)
                DF_After = DF_Storm[DF_Storm["Time(Z)"] >= After].reset_index(drop=True)
# Choose the Part with the Lowest Min SLP to Keep
                if numpy.min(DF_After["SLP(hPa)"]) <= numpy.min(DF_Before["SLP(hPa)"]) and \
                len(DF_After) > len(DF_Before):
                    DF_Storm_Updated = DF_After
                else:
                    DF_Storm_Updated = DF_Before
                Track_Continuous = 1
# Repeat Until No More Track Discontinuity
    if Track_Continuous == 0:
        DF_Storm_Updated = DF_Storm.copy()
    return (DF_Storm_Updated, Track_Discon)

In [22]:
# Fix Storms That Have Time Discontinuity or Track Discontinuity
def DF_Fixing(DF_Storm_Orig, Track_Discons):
    Bottom_Right = False
    Top_Right = False
    Bottom_Left = False
    Top_Left = False
    Bottom_Right_t, Top_Right_t, Bottom_Left_t = [], [], []
    ET_Complete = False
#
# Check For Time Discontinuity
    DF_Storm = Time_Discontinuity(DF_Storm_Orig)
# Repeat to Check Again
    DF_Storm = Time_Discontinuity(DF_Storm)
    DF_Storm = Time_Discontinuity(DF_Storm)
#
# Check For Track Discontinuity
    DF_Storm, Track_Discon = Track_Discontinuity(DF_Storm)
    if len(Track_Discon) > 0:
        Track_Discons.append(Track_Discon)
# Repeat to Check Again
    DF_Storm, Track_Discon = Track_Discontinuity(DF_Storm)
    if len(Track_Discon) > 0:
        Track_Discons.append(Track_Discon)
    DF_Storm, Track_Discon = Track_Discontinuity(DF_Storm)
    if len(Track_Discon) > 0:
        Track_Discons.append(Track_Discon)
    return (DF_Storm, Track_Discons)

In [23]:
# Function to Find Whether a Storm Has Completed ET, and Start and End Time of ET
def ET_Function(DF_Storm):
    Tropical_Begin = False
    ET_Begin = False
    ET_Complete = False
    Bottom_Right_t, Top_Right_t, Bottom_Left_t = [], [], []
    Trop_Begin_t, ET_Begin_t, ET_Compl_t, Trans_Type = -728, -728, -728, -728
    for t in range(len(DF_Storm)):
        if ET_Complete == False:
            if Tropical_Begin == False:
# Storm Must First Be Tropical
                if DF_Storm["B"][t] <= 15 and DF_Storm["VLT"][t] >= 0:
                    Tropical_Begin = True
                    Trop_Begin_t = t
                    Bottom_Right_t.append(t)
            else:
# Storm Must Be Tropical For At Least 48 Hours
                if len(Bottom_Right_t) < 8:
                    if DF_Storm["B"][t] <= 15 and DF_Storm["VLT"][t] >= 0:
                        Bottom_Right_t.append(t)
#
# Enter Next Stage If Storm Remains Tropical For At Least 48 Hours
                else:
                    if ET_Begin == False:
# If Storm Enters Top Right Quadrant = ET Begin
                        if DF_Storm["B"][t] > 15 and DF_Storm["VLT"][t] >= 0:
                            ET_Begin = True
                            ET_Begin_t = t
                            Top_Right_t.append(t)
# If Storm Enters Bottom Left Quadrant = ET Begin
                        elif DF_Storm["B"][t] <= 15 and DF_Storm["VLT"][t] < 0:
                            ET_Begin = True
                            ET_Begin_t = t
                            Bottom_Left_t.append(t)
# If Storm Directly Enters Top Left Quadrant = ET Complete
                        elif DF_Storm["B"][t] > 15 and DF_Storm["VLT"][t] < 0:
                            ET_Complete = True
                            ET_Begin_t, ET_Compl_t = t, t
#
# After ET Initiation
                    elif ET_Begin == True:
                        if len(Top_Right_t) > 0 and len(Top_Right_t) >= len(Bottom_Left_t):
# If Storm Remains in Top Right Quadrant
                            if DF_Storm["B"][t] > 15 and DF_Storm["VLT"][t] >= 0:
                                Top_Right_t.append(t)
# If Storm Enters Top Left Quadrant = ET Complete
                            elif DF_Storm["VLT"][t] < 0:
                                if DF_Storm["B"][t] > 10:
                                    ET_Complete = True
                                    ET_Compl_t = t
# If Storm Enters Bottom Left Quadrant
                                else:
                                    Bottom_Left_t.append(t)
                            elif DF_Storm["B"][t] <= 15 and DF_Storm["VLT"][t] >= 0:
# If Storm Returns to Bottom Right Quadrant For Over 24 Hours
                                if t > Top_Right_t[-1] + 4:
                                    ET_Begin = False
                                    ET_Begin_t = -728
                                    Top_Right_t = []
                        elif len(Bottom_Left_t) > 0:
# If Storm Remains in Bottom Left Quadrant
                            if DF_Storm["B"][t] <= 15 and DF_Storm["VLT"][t] < 0:
                                Bottom_Left_t.append(t)
# If Storm Enters Top Left Quadrant = ET Complete
                            elif DF_Storm["B"][t] > 15:
                                if DF_Storm["VLT"][t] <= 0:
                                    ET_Complete = True
                                    ET_Compl_t = t
# If Storm Enters Top Right Quadrant
                                else:
                                    Top_Right_t.append(t)
                            elif DF_Storm["B"][t] <= 15 and DF_Storm["VLT"][t] >= 0:
# If Storm Returns to Bottom Right Quadrant For Over 24 Hours
                                if t-4 > Bottom_Left_t[len(Bottom_Left_t)-1]:
                                    ET_Begin = False
                                    ET_Begin_t = -728
                                    Bottom_Left_t = []
#
# If Storm Did Not Spend At Least 48 Hours as Tropical
    if len(Bottom_Right_t) < 8:
        Trans_Type = -728
# If ET Never Begun:
    elif ET_Compl_t == -728:
        if ET_Begin_t == -728:
            Trans_Type = 0
# If ET Began But Did Not Complete:
        else:
            Trans_Type = 1
    else:
        Trans_Type = 2
    return (Trans_Type, Trop_Begin_t, ET_Begin_t, ET_Compl_t)

In [24]:
# Apply Function For Control DataFrames
Control_Data_V2, Control_ET_V1 = Storm_Info_DF(Control_Data_V1, Control_Codes_V1)

In [25]:
# Apply Function For RCP4.5 DataFrames
RCP45_Data_V2, RCP45_ET_V1 = Storm_Info_DF(RCP45_Data_V1, RCP45_Codes_V1)

In [26]:
# Apply Function For RCP8.5 DataFrames
RCP85_Data_V2, RCP85_ET_V1 = Storm_Info_DF(RCP85_Data_V1, RCP85_Codes_V1)

In [27]:
# Add Storm Phase into DataFrame
def Storm_Phase_Info(Data_DF_Orig, ET_DF):
    Data_DF = Data_DF_Orig.copy()
    Storm_Phase_List = []
    for i in range(len(ET_DF)):
        DF_Storm = Data_DF[Data_DF["Code"] == ET_DF["Code"][i]].reset_index(drop=True)
# Find Storm Phase Based on ET Start Time and ET End Time
        Storm_Phase = Find_Storm_Phase(DF_Storm, ET_DF["Trop Begin Time"][i], ET_DF["ET Begin Time"][i], ET_DF["ET Complete Time"][i], ET_DF["Trans Type"][i])
        for j in range(len(Storm_Phase)):
            Storm_Phase_List.append(Storm_Phase[j])
    Data_DF["Storm Phase"] = Storm_Phase_List
    return (Data_DF)

In [28]:
# Function to Find Storm Phase
def Find_Storm_Phase(DF_Storm, Trop_Begin_Time, ET_Begin_Time, ET_Compl_Time, Trans_Type):
    Time = list(DF_Storm["Time(Z)"])
    Storm_Phase = []
    for k in range(len(DF_Storm)):
# If Before First Tropical = Pre-Tropical
# If Before ET Start = Tropical
# If Between ET Start and ET End = Transitioning
# If After ET End = Extratropical
        if Time[k] < Trop_Begin_Time:
            Storm_Phase.append("Pre-Tropical")
        else:
            if Trans_Type == 0:
                Storm_Phase.append("Tropical")
            elif Trans_Type == 1:
                if Time[k] < ET_Begin_Time:
                    Storm_Phase.append("Tropical")
                else:
                    Storm_Phase.append("Transition")
            else:
                if Time[k] < ET_Begin_Time:
                    Storm_Phase.append("Tropical")
                elif Time[k] < ET_Compl_Time:
                    Storm_Phase.append("Transition")
                else:
                    Storm_Phase.append("Extratropical")
    return (Storm_Phase)

In [29]:
# Apply Function
Control_Data_V3 = Storm_Phase_Info(Control_Data_V2, Control_ET_V1)
RCP45_Data_V3 = Storm_Phase_Info(RCP45_Data_V2, RCP45_ET_V1)
RCP85_Data_V3 = Storm_Phase_Info(RCP85_Data_V2, RCP85_ET_V1)

In [30]:
# Add Lat, Lon, SLP Into ET DataFrame
def ET_Variables(Main_DF, ET_DF_Orig):
    ET_DF = ET_DF_Orig.copy()
    Array = numpy.zeros((21, len(ET_DF)))
    Trop_Peak_Times = []
    Peak_Times = []
    Genesis_Times = []
    Final_Times = []
    for i in range(len(ET_DF)):
        Code = ET_DF["Code"][i]
        Trans_Type, ET_Begin_Time, ET_Compl_Time, Trop_Begin_Time = \
        ET_DF["Trans Type"][i], ET_DF["ET Begin Time"][i], ET_DF["ET Complete Time"][i], ET_DF["Trop Begin Time"][i]
        Storm = Main_DF[Main_DF["Code"] == Code].reset_index(drop=True)
#
# Find Lat, Lon, Time at Storm Peak
        Min_SLP = numpy.min(Storm["SLP(hPa)"])
        Storm_Peak = Storm[Storm["SLP(hPa)"] == Min_SLP]
        Array[0][i] = Min_SLP
        Array[1][i] = float(Storm_Peak["Lon"].iloc[0])
        Array[2][i] = float(Storm_Peak["Lat"].iloc[0])
        Peak_Times.append(Storm_Peak["Time(Z)"].iloc[0])
#
# Find Lat, Lon, Time at Tropical Phase Peak
        Storm_Trop = Storm[Storm["Storm Phase"] == "Tropical"]
        Trop_Min_SLP = numpy.min(Storm_Trop["SLP(hPa)"])
        Trop_Peak = Storm[Storm["SLP(hPa)"] == Trop_Min_SLP]
        if len(Storm_Trop) > 0:
            Array[3][i] = Trop_Min_SLP
            Array[4][i] = float(Trop_Peak["Lon"].iloc[0])
            Array[5][i] = float(Trop_Peak["Lat"].iloc[0])
            Trop_Peak_Times.append(Trop_Peak["Time(Z)"].iloc[0])
        else:
            Array[3][i] = numpy.nan
            Array[4][i] = numpy.nan
            Array[5][i] = numpy.nan
            Trop_Peak_Times.append(numpy.nan)
#
# Find SLP, Lon, Lat at Genesis
        Array[6][i] = float(Storm["SLP(hPa)"].iloc[0])
        Array[7][i] = float(Storm["Lon"].iloc[0])
        Array[8][i] = float(Storm["Lat"].iloc[0])
        Genesis_Times.append(Storm["Time(Z)"].iloc[0])
#
# Find SLP, Lon, Lat at Final
        Array[9][i] = float(Storm["SLP(hPa)"].iloc[-1])
        Array[10][i] = float(Storm["Lon"].iloc[-1])
        Array[11][i] = float(Storm["Lat"].iloc[-1])
        Final_Times.append(Storm["Time(Z)"].iloc[-1])
#
#
# Find SLP, Lon, Lat at Tropical Begin
        Trop_Begin = Storm[Storm["Time(Z)"] == Trop_Begin_Time]
        if len(Trop_Begin) > 0:
            Array[12][i] = float(Trop_Begin["SLP(hPa)"].iloc[0])
            Array[13][i] = float(Trop_Begin["Lon"].iloc[0])
            Array[14][i] = float(Trop_Begin["Lat"].iloc[0])
        else:
            Array[12][i], Array[13][i], Array[14][i] = numpy.nan, numpy.nan, numpy.nan
#
# Find SLP, Lon, Lat at ET Begin
        if Trans_Type >= 1:
            Begin = Storm[Storm["Time(Z)"] == ET_Begin_Time]
            Array[15][i] = float(Begin["SLP(hPa)"].iloc[0])
            Array[16][i] = float(Begin["Lon"].iloc[0])
            Array[17][i] = float(Begin["Lat"].iloc[0])
        else:
            Array[15][i], Array[16][i], Array[17][i] = numpy.nan, numpy.nan, numpy.nan
#
# Find SLP, Lon, Lat at ET Complete
        if Trans_Type == 2:
            Complete = Storm[Storm["Time(Z)"] == ET_Compl_Time]
            Array[18][i] = float(Complete["SLP(hPa)"].iloc[0])
            Array[19][i] = float(Complete["Lon"].iloc[0])
            Array[20][i] = float(Complete["Lat"].iloc[0])
        else:
            Array[18][i], Array[19][i], Array[20][i] = numpy.nan, numpy.nan, numpy.nan
#
# Append Into ET DF
    ET_DF["Peak Time"] = Peak_Times
    ET_DF["Trop Peak Time"] = Trop_Peak_Times
    ET_DF["Genesis Time"] = Genesis_Times
    ET_DF["Final Time"] = Final_Times
    Cols = ["Peak SLP", "Peak Lon", "Peak Lat", "Trop Peak SLP", "Trop Peak Lon", "Trop Peak Lat", \
    "Genesis SLP", "Genesis Lon", "Genesis Lat", "Final SLP", "Final Lon", "Final Lat", "Trop Begin SLP", "Trop Begin Lon", "Trop Begin Lat", \
    "ET Begin SLP", "ET Begin Lon", "ET Begin Lat", "ET Complete SLP", "ET Complete Lon", "ET Complete Lat"]
    ET_DF_Cols = pandas.DataFrame(Array.T, columns=Cols, index=ET_DF.index)
    ET_DF = pandas.concat([ET_DF, ET_DF_Cols], axis=1)
# Rearrange Columns
    ET_DF = Move_Col(ET_DF, "Peak Time", "Peak SLP")
    ET_DF = Move_Col(ET_DF, "Trop Peak Time", "Trop Peak SLP")
    ET_DF = Move_Col(ET_DF, "Genesis Time", "Genesis SLP")
    ET_DF = Move_Col(ET_DF, "Final Time", "Final SLP")
    ET_DF = Move_Col(ET_DF, "Trop Begin Time", "Trop Peak SLP")
    ET_DF = Move_Col(ET_DF, "ET Begin Time", "ET Begin SLP")
    ET_DF = Move_Col(ET_DF, "ET Complete Time", "ET Complete SLP")
    return (ET_DF)

In [31]:
def Move_Col(DF, Move_Col, Ref_Col):
    Col = DF.pop(Move_Col)
    Index = DF.columns.get_loc(Ref_Col)
    DF.insert(Index, Move_Col, Col)
    return (DF)

In [32]:
# Apply Function
Control_ET_V2 = ET_Variables(Control_Data_V3, Control_ET_V1)
RCP45_ET_V2 = ET_Variables(RCP45_Data_V3, RCP45_ET_V1)
RCP85_ET_V2 = ET_Variables(RCP85_Data_V3, RCP85_ET_V1)

In [33]:
print (len(Control_ET_V2), len(RCP45_ET_V2), len(RCP85_ET_V2))

943 746 650


In [34]:
pandas.set_option("display.max_columns", None)
Control_ET_V2

,Code,Orig Code,Ensemble,Trans Type,Peak Time,Peak SLP,Peak Lon,Peak Lat,Trop Peak Time,Trop Begin Time,Trop Peak SLP,Trop Peak Lon,Trop Peak Lat,Genesis Time,Genesis SLP,Genesis Lon,Genesis Lat,Final Time,Final SLP,Final Lon,Final Lat,Trop Begin SLP,Trop Begin Lon,Trop Begin Lat,ET Begin Time,ET Begin SLP,ET Begin Lon,ET Begin Lat,ET Complete Time,ET Complete SLP,ET Complete Lon,ET Complete Lat
0,TC190001,0024,A,2,1900-08-31 12:00:00,998.93,-42.40,41.54,1900-08-27 06:00:00,1900-08-25 00:00:00,999.11,-68.55,31.19,1900-08-25 00:00:00,1010.17,-74.91,22.04,1900-09-02 06:00:00,1010.34,-25.50,46.00,1010.17,-74.91,22.04,1900-08-28 00:00:00,1001.52,-65.34,32.70,1900-09-01 00:00:00,1006.23,-37.00,42.75
1,TC190002,0026,A,2,1900-09-03 06:00:00,933.33,-53.61,30.15,1900-09-03 06:00:00,1900-08-29 06:00:00,933.33,-53.61,30.15,1900-08-26 12:00:00,1003.52,-7.32,15.81,1900-09-10 06:00:00,978.17,-16.15,55.43,1003.29,-27.73,17.28,1900-09-04 06:00:00,943.55,-50.41,35.84,1900-09-10 06:00:00,978.17,-16.15,55.43
2,TC190003,0028,A,1,1900-09-09 12:00:00,972.24,-51.89,48.03,1900-09-07 00:00:00,1900-09-04 18:00:00,993.64,-64.50,40.87,1900-09-04 12:00:00,998.46,-68.21,35.95,1900-09-11 06:00:00,996.61,-38.50,46.25,997.93,-67.63,36.53,1900-09-08 06:00:00,993.82,-60.67,40.65,NaT,NaN,NaN,NaN
3,TC190004,0029,A,1,1900-09-16 06:00:00,988.35,-49.10,43.53,1900-09-14 18:00:00,1900-09-11 12:00:00,997.38,-56.81,32.09,1900-09-11 12:00:00,1009.35,-46.84,17.12,1900-09-16 06:00:00,988.35,-49.10,43.53,1009.35,-46.84,17.12,1900-09-15 06:00:00,993.79,-54.45,35.06,NaT,NaN,NaN,NaN
4,TC190005,0032,A,-728,1900-10-07 00:00:00,980.38,-54.41,45.39,NaT,1900-09-11 12:00:00,NaN,NaN,NaN,1900-10-04 18:00:00,1006.33,-70.34,33.78,1900-10-07 12:00:00,989.32,-49.24,48.76,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaT,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
938,TC198908,1410,C,0,1989-09-09 18:00:00,1000.49,-36.54,32.93,1989-09-09 18:00:00,1989-09-09 12:00:00,1000.49,-36.54,32.93,1989-09-09 12:00:00,1002.62,-36.24,33.22,1989-09-11 18:00:00,1012.12,-43.28,32.06,1002.62,-36.24,33.22,NaT,NaN,NaN,NaN,NaT,NaN,NaN,NaN
939,TC198909,1415,C,-728,1989-10-05 00:00:00,986.25,-15.25,54.75,NaT,1989-09-29 18:00:00,NaN,NaN,NaN,1989-09-29 18:00:00,1007.42,-59.33,32.85,1989-10-05 00:00:00,986.25,-15.25,54.75,1007.42,-59.33,32.85,NaT,NaN,NaN,NaN,NaT,NaN,NaN,NaN
940,TC198910,1417,C,0,1989-10-16 12:00:00,994.59,-96.37,23.33,1989-10-16 12:00:00,1989-10-14 06:00:00,994.59,-96.37,23.33,1989-10-14 06:00:00,1011.41,-88.80,26.69,1989-10-18 00:00:00,1011.47,-99.50,27.25,1011.41,-88.80,26.69,NaT,NaN,NaN,NaN,NaT,NaN,NaN,NaN
941,TC198911,1419,C,2,1989-10-24 06:00:00,995.38,-29.77,40.22,1989-10-24 06:00:00,1989-10-23 06:00:00,995.38,-29.77,40.22,1989-10-22 18:00:00,998.53,-35.77,37.78,1989-11-01 12:00:00,1010.12,-20.00,30.00,998.03,-32.57,37.26,1989-10-26 12:00:00,1001.83,-26.73,37.33,1989-10-30 18:00:00,1004.53,-26.75,28.50


In [35]:
# Filter Out Storms That Have Not Been Tropical For At Least 48 Hours
def Tropical_Filter(Data_DF_Orig, ET_DF_Orig, Codes_DF_Orig):
    Filtered_ET_0 = ET_DF_Orig[ET_DF_Orig["Trans Type"] > -728].reset_index(drop=True)
# Filter Out Storms With Tropical Peak SLP >= 1000hPa
    Filtered_ET = Filtered_ET_0[Filtered_ET_0["Trop Peak SLP"] < 1000].reset_index(drop=True)
    Code_List = Filtered_ET["Code"]
    Filtered_DF = Data_DF_Orig[Data_DF_Orig["Code"].isin(Code_List)].reset_index(drop=True)
    Filtered_Codes = Codes_DF_Orig[Codes_DF_Orig["New Code"].isin(Code_List)].reset_index(drop=True)
    return (Filtered_DF, Filtered_ET, Filtered_Codes)

In [36]:
# Apply Function
Control_Data_V4, Control_ET_V4, Control_Codes_V4 = Tropical_Filter(Control_Data_V3, Control_ET_V2, Control_Codes_V1)
RCP45_Data_V4, RCP45_ET_V4, RCP45_Codes_V4 = Tropical_Filter(RCP45_Data_V3, RCP45_ET_V2, RCP45_Codes_V1)
RCP85_Data_V4, RCP85_ET_V4, RCP85_Codes_V4 = Tropical_Filter(RCP85_Data_V3, RCP85_ET_V2, RCP85_Codes_V1)

In [37]:
print (len(Control_ET_V4), len(RCP45_ET_V4), len(RCP85_ET_V4))

595 413 360


In [38]:
# Count Trans Type
def Trans_Type_Percent(Series):
    Percents = Series.value_counts(normalize=True).reindex([0, 1, 2], fill_value=0)
    return (Percents)

In [39]:
# Calculate Percentages of ET Completion
ET_Percents = pandas.DataFrame({"Control": Trans_Type_Percent(Control_ET_V4["Trans Type"]), \
"RCP4.5": Trans_Type_Percent(RCP45_ET_V4["Trans Type"]), \
"RCP8.5": Trans_Type_Percent(RCP85_ET_V4["Trans Type"])})
ET_Percents.index = ["No ET", "ET Incomplete", "ET Complete"]
ET_Percents

,Control,RCP4.5,RCP8.5
No ET,0.272269,0.251816,0.233333
ET Incomplete,0.218487,0.220339,0.188889
ET Complete,0.509244,0.527845,0.577778


In [40]:
# Assign Names To Each Cyclone Code
Control_Data_V5, Control_ET_V5, Control_Codes_V5 = Assign_Name(Name_List, Control_Data_V4, Control_ET_V4, Control_Codes_V4)
RCP45_Data_V5, RCP45_ET_V5, RCP45_Codes_V5 = Assign_Name(Name_List, RCP45_Data_V4, RCP45_ET_V4, RCP45_Codes_V4)
RCP85_Data_V5, RCP85_ET_V5, RCP85_Codes_V5 = Assign_Name(Name_List, RCP85_Data_V4, RCP85_ET_V4, RCP85_Codes_V4)

In [41]:
# Filter For Storms That Completed ET Transition
def ET_Filter(Data_DF_Orig, ET_DF_Orig):
    ET_DF = ET_DF_Orig[ET_DF_Orig["Trans Type"] == 2].reset_index(drop=True)
    Code_List = ET_DF["Code"]
    Filtered_DF = Data_DF_Orig[Data_DF_Orig["Code"].isin(Code_List)].reset_index(drop=True)
    return (Filtered_DF, ET_DF)

In [42]:
# Apply Function
Control_Data_V6, Control_ET_V6 = ET_Filter(Control_Data_V5, Control_ET_V5)
RCP45_Data_V6, RCP45_ET_V6 = ET_Filter(RCP45_Data_V5, RCP45_ET_V5)
RCP85_Data_V6, RCP85_ET_V6 = ET_Filter(RCP85_Data_V5, RCP85_ET_V5)

In [43]:
print (len(Control_ET_V6), len(RCP45_ET_V6), len(RCP85_ET_V6))

303 218 208


In [44]:
# Filter For Storms With Trop Peak SLP <= 1000, ET Begin SLP <= 1005 and ET Complete SLP <= 1005
def SLP_Filter(Data_DF_Orig, ET_DF_Orig):
    ET_DF = ET_DF_Orig[(ET_DF_Orig["Trop Peak SLP"] <= 1000) & (ET_DF_Orig["ET Begin SLP"] <= 1005) & \
    (ET_DF_Orig["ET Complete SLP"] <= 1005)].reset_index(drop=True)
    Code_List = ET_DF["Code"]
    Filtered_DF = Data_DF_Orig[Data_DF_Orig["Code"].isin(Code_List)].reset_index(drop=True)
    return (Filtered_DF, ET_DF)

In [45]:
# Apply Function
Control_Data_V7, Control_ET_V7 = SLP_Filter(Control_Data_V6, Control_ET_V6)
RCP45_Data_V7, RCP45_ET_V7 = SLP_Filter(RCP45_Data_V6, RCP45_ET_V6)
RCP85_Data_V7, RCP85_ET_V7 = SLP_Filter(RCP85_Data_V6, RCP85_ET_V6)

In [46]:
print (len(Control_ET_V7), len(RCP45_ET_V7), len(RCP85_ET_V7))

265 192 186


In [47]:
# Filter For Storms Based on ET Location
def ET_Loc_Filter(Data_DF_Orig, ET_DF_Orig):
# Filter Out ETs Near European Coast
    ET_DF_Filt = ET_DF_Orig[(ET_DF_Orig["ET Complete Lon"] < -12)]
# Filter Out ETs Deep Inland Within North America
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Begin Lat"] < 31) | (ET_DF_Filt["ET Begin Lon"] > -83)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Begin Lat"] < 34) | (ET_DF_Filt["ET Begin Lon"] > -81)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Begin Lat"] < 36) | (ET_DF_Filt["ET Begin Lon"] > -78)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Begin Lat"] < 40) | (ET_DF_Filt["ET Begin Lon"] > -76)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Begin Lat"] < 45) | (ET_DF_Filt["ET Begin Lon"] > -72)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Begin Lat"] < 49) | (ET_DF_Filt["ET Begin Lon"] > -68)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Begin Lat"] < 52) | (ET_DF_Filt["ET Begin Lon"] > -62)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Complete Lat"] < 31) | (ET_DF_Filt["ET Complete Lon"] > -83)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Complete Lat"] < 34) | (ET_DF_Filt["ET Complete Lon"] > -81)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Complete Lat"] < 36) | (ET_DF_Filt["ET Complete Lon"] > -78)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Complete Lat"] < 40) | (ET_DF_Filt["ET Complete Lon"] > -76)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Complete Lat"] < 45) | (ET_DF_Filt["ET Complete Lon"] > -72)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Complete Lat"] < 49) | (ET_DF_Filt["ET Complete Lon"] > -68)]
    ET_DF_Filt = ET_DF_Filt[(ET_DF_Filt["ET Complete Lat"] < 52) | (ET_DF_Filt["ET Complete Lon"] > -62)]
    ET_DF = ET_DF_Filt.reset_index(drop=True)
    Code_List = ET_DF["Code"]
    Filtered_DF = Data_DF_Orig[Data_DF_Orig["Code"].isin(Code_List)].reset_index(drop=True)
    return (Filtered_DF, ET_DF)

In [48]:
# Apply Function
Control_Data_V8, Control_ET_V8 = ET_Loc_Filter(Control_Data_V7, Control_ET_V7)
RCP45_Data_V8, RCP45_ET_V8 = ET_Loc_Filter(RCP45_Data_V7, RCP45_ET_V7)
RCP85_Data_V8, RCP85_ET_V8 = ET_Loc_Filter(RCP85_Data_V7, RCP85_ET_V7)

In [49]:
print (len(Control_ET_V8), len(RCP45_ET_V8), len(RCP85_ET_V8))

226 164 164


In [50]:
# Filter For Storms Based on Phase Space Parameters
def Phase_Space_Filter(Data_DF_Orig, ET_DF_Orig):
    Unsure_Codes = ET_DF_Orig["Code"]
    Codes_Remove = []
    for i in range(len(Unsure_Codes)):
        DF_Storm = Find_Storm(Data_DF_Orig, Unsure_Codes[i])
        DF_Storm_TC = DF_Storm[DF_Storm["Storm Phase"] == "Tropical"].reset_index(drop=True)
# Remove Storms With Trop Peak Outside Bottom Right Quadrant
        Trop_Peak_SLP = numpy.min(DF_Storm_TC["SLP(hPa)"])
        for j in range(len(DF_Storm_TC)):
            if DF_Storm_TC["SLP(hPa)"][j] == Trop_Peak_SLP:
                if (DF_Storm_TC["B"][j] > 20) or (DF_Storm_TC["VLT"][j] < 0):
                    if Unsure_Codes[i] not in Codes_Remove:
                        Codes_Remove.append(Unsure_Codes[i])
# Remove Filtered Out Storms
    ET_DF = ET_DF_Orig[~ET_DF_Orig["Code"].isin(Codes_Remove)].reset_index(drop=True)
    Code_List = ET_DF["Code"]
    Filtered_DF = Data_DF_Orig[Data_DF_Orig["Code"].isin(Code_List)].reset_index(drop=True)
#    print (Codes_Remove)
    return (Filtered_DF, ET_DF)

In [51]:
# Apply Function
Control_Data_V10, Control_ET_V10 = Phase_Space_Filter(Control_Data_V8, Control_ET_V8)
RCP45_Data_V10, RCP45_ET_V10 = Phase_Space_Filter(RCP45_Data_V8, RCP45_ET_V8)
RCP85_Data_V10, RCP85_ET_V10 = Phase_Space_Filter(RCP85_Data_V8, RCP85_ET_V8)

In [52]:
print (len(Control_ET_V10), len(RCP45_ET_V10), len(RCP85_ET_V10))

226 161 161


In [53]:
# Filter For Storms Based on Tropical Peak SLP and Tropical Duration
def Peak_SLP_Filter(Data_DF_Orig, ET_DF_Orig):
    Unsure_Codes = list(ET_DF_Orig["Code"])
    Codes_Remove = []
    for i in range(len(Unsure_Codes)):
        DF_Storm = Find_Storm(Data_DF_Orig, Unsure_Codes[i])
        DF_Storm_Trop = DF_Storm[DF_Storm["Storm Phase"] == "Tropical"].reset_index(drop=True)
# Filter Out Storms With Genesis SLP - Tropical Peak SLP < 10:
        if DF_Storm_Trop["SLP(hPa)"][0] - numpy.min(DF_Storm_Trop["SLP(hPa)"]) < 10 and Unsure_Codes[i] not in Codes_Remove:
            Codes_Remove.append(Unsure_Codes[i])
# Filter Out Storms With Tropical Duration < 72 hours:
        elif len(DF_Storm_Trop["SLP(hPa)"]) < 12 and Unsure_Codes[i] not in Codes_Remove:
            Codes_Remove.append(Unsure_Codes[i])
# Remove Filtered Out Storms
    ET_DF = ET_DF_Orig[~ET_DF_Orig["Code"].isin(Codes_Remove)].reset_index(drop=True)
    Code_List = ET_DF["Code"]
    Filtered_DF = Data_DF_Orig[Data_DF_Orig["Code"].isin(Code_List)].reset_index(drop=True)
#    print (Codes_Remove)
    return (Filtered_DF, ET_DF)

In [54]:
# Apply Function
Control_Data_V11, Control_ET_V11 = Peak_SLP_Filter(Control_Data_V10, Control_ET_V10)
RCP45_Data_V11, RCP45_ET_V11 = Peak_SLP_Filter(RCP45_Data_V10, RCP45_ET_V10)
RCP85_Data_V11, RCP85_ET_V11 = Peak_SLP_Filter(RCP85_Data_V10, RCP85_ET_V10)

In [55]:
print (len(Control_ET_V11), len(RCP45_ET_V11), len(RCP85_ET_V11))

174 113 108


In [56]:
Incomplete_Codes = ["TC191013", "TC191309", "TC192010", "TC192211", "TC192306", "TC192904", "TC193601", "TC194802", "TC195107", "TC195309", "TC195808", \
"TC200004", "TC201309", "TC202406", "TC202508", "TC202601", "TC203303", "TC203803", "TC204304", "TC206201", "TC206601", "TC207801", "TC208806", \
"TC210005", "TC211501", "TC212201", "TC213506", "TC214601", "TC214805", "TC215101", "TC215702", "TC216202", "TC216209", "TC218204"]

In [57]:
# Filter Out Storms With Incomplete Composite Data
def Incomplete_Filter(Data_DF_Orig, ET_DF_Orig, Incomplete_Codes):
    Unsure_Codes = list(ET_DF_Orig["Code"])
    Codes_Remove = []
    for i in range(len(Unsure_Codes)):
        if Unsure_Codes[i] in Incomplete_Codes:
            Codes_Remove.append(Unsure_Codes[i])
    ET_DF = ET_DF_Orig[~ET_DF_Orig["Code"].isin(Codes_Remove)].reset_index(drop=True)
    Code_List = ET_DF["Code"]
    Filtered_DF = Data_DF_Orig[Data_DF_Orig["Code"].isin(Code_List)].reset_index(drop=True)
#    print (Codes_Remove)
    return (Filtered_DF, ET_DF)

In [58]:
# Apply Function
Control_Data_V12, Control_ET_V12 = Incomplete_Filter(Control_Data_V11, Control_ET_V11, Incomplete_Codes)
RCP45_Data_V12, RCP45_ET_V12 = Incomplete_Filter(RCP45_Data_V11, RCP45_ET_V11, Incomplete_Codes)
RCP85_Data_V12, RCP85_ET_V12 = Incomplete_Filter(RCP85_Data_V11, RCP85_ET_V11, Incomplete_Codes)

In [59]:
print (len(Control_ET_V12), len(RCP45_ET_V12), len(RCP85_ET_V12))

174 109 100


In [60]:
# Output DF to csv File
def Output_File(DF, Model, Type):
    File_Name = str(Model+'_'+Type+'_Output.csv')
    DF.to_csv(Output_Diri+File_Name)

In [61]:
# Output Control Files
Output_File(Control_Data_V5, "Control", "Data_DatasetA")
Output_File(Control_Data_V6, "Control", "Data_SubsetB")
Output_File(Control_Data_V12, "Control", "Data_SubsetC")
Output_File(Control_ET_V5, "Control", "ET_DatasetA")
Output_File(Control_ET_V6, "Control", "ET_SubsetB")
Output_File(Control_ET_V12, "Control", "ET_SubsetC")
Output_File(Control_Codes_V5, "Control", "Codes")

In [62]:
# Output RCP4.5 Files
Output_File(RCP45_Data_V5, "RCP45", "Data_DatasetA")
Output_File(RCP45_Data_V6, "RCP45", "Data_SubsetB")
Output_File(RCP45_Data_V12, "RCP45", "Data_SubsetC")
Output_File(RCP45_ET_V5, "RCP45", "ET_DatasetA")
Output_File(RCP45_ET_V6, "RCP45", "ET_SubsetB")
Output_File(RCP45_ET_V12, "RCP45", "ET_SubsetC")
Output_File(RCP45_Codes_V5, "RCP45", "Codes")

In [63]:
# Output RCP8.5 Files
Output_File(RCP85_Data_V5, "RCP85", "Data_DatasetA")
Output_File(RCP85_Data_V6, "RCP85", "Data_SubsetB")
Output_File(RCP85_Data_V12, "RCP85", "Data_SubsetC")
Output_File(RCP85_ET_V5, "RCP85", "ET_DatasetA")
Output_File(RCP85_ET_V6, "RCP85", "ET_SubsetB")
Output_File(RCP85_ET_V12, "RCP85", "ET_SubsetC")
Output_File(RCP85_Codes_V5, "RCP85", "Codes")